# Задача генерации текста — DialogSum

## 01 - Imports

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import evaluate

import torch
import transformers
import vllm

from vllm import LLM, SamplingParams

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB"
    )

## 02 — Dataset loading

Загрузим, а после посмотрим на данные ,проведем EDA

In [ ]:
%pip install kaggle -q

In [ ]:
!kaggle --version

In [ ]:
!kaggle datasets download -d marawanxmamdouh/dialogsum

In [ ]:
import zipfile

with zipfile.ZipFile("dialogsum.zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [ ]:
!ls data/CSV

In [ ]:
train_df = pd.read_csv("data/CSV/train.csv")
valid_df = pd.read_csv("data/CSV/validation.csv")
test_df = pd.read_csv("data/CSV/test.csv")

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test:", test_df.shape)

In [ ]:
train_df.head()

## 03 — EDA

Проверим данные на наличчие пропусков и дубликатов

In [ ]:
train_df.isna().sum()

In [ ]:
valid_df.isna().sum()

In [ ]:
test_df.isna().sum()

In [ ]:
train_df.duplicated().sum()

In [ ]:
train_df.duplicated(subset='dialogue').sum()

In [ ]:
train_df[train_df.duplicated(subset=['dialogue'], keep=False)]

In [ ]:
test_df

In [ ]:
print(test_df.shape)
print(test_df['dialogue'].nunique())
print(test_df.duplicated(subset='dialogue').sum())

In [ ]:
test_df.duplicated(subset='dialogue').sum()

In [ ]:
dup_dialogue = test_df.loc[
    test_df['dialogue'].duplicated(keep=False),
    'dialogue'
].iloc[0]

test_df[test_df['dialogue'] == dup_dialogue][
    ['id', 'dialogue', 'summary', 'topic']
]

В обучающей выборке обнаружено 2 повторяющихся диалога. При этом соответствующие summary и topic различаются, то есть это не полные дубликаты, а одинаковые входные тексты с разными вариантами разметки. Поскольку таких случаев крайне мало относительно размера обучающей выборки, было решено оставить их без изменений.

В тестовой выборке наблюдаем одинаковые примеры диологов и разные summary и topic, то есть несколько правильных вариантов

Проверим наличие дубликатов во всех выборках

In [ ]:
len(set(train_df['dialogue']) & set(test_df['dialogue']))

In [ ]:
len(set(train_df['dialogue']) & set(valid_df['dialogue']))

In [ ]:
len(set(valid_df['dialogue']) & set(test_df['dialogue']))

Посчитаем колличество слов в текстах

In [ ]:
train_df['count_words'] = train_df['dialogue'].apply(lambda x: len(x.split()))

In [ ]:
train_df['count_words'].describe()

Видно что в диалогах среднее колличество слов равно 131, медиана равна 116, минимальное значение - 40 , а вот максимальное 985 ,стоит проверить

In [ ]:
train_df[train_df['count_words'] == max(train_df['count_words'])]

Максимальная длина диалога составляет 985 слов, что существенно превышает 99-й перцентиль (375 слов). Ручная проверка показала, что пример является содержательно связным длинным диалогом, а не очевидной ошибкой или склейкой данных, поэтому он сохраняется в датасете.

In [ ]:
train_df['count_words'].quantile([0.90,0.95,0.99])

Так же посчитаем на сколько сильно текст сжат после суммаризации

In [ ]:
train_df['summary_words_count'] = train_df['summary'].apply(lambda x: len(x.split()))

In [ ]:
train_df['summary_words_count'].describe()

In [ ]:
train_df['summary_words_count'].quantile([0.90,0.95,0.99])

Для наглядности посчитаем во сколько раз суммаризованный текст меньше исходного

In [ ]:
train_df['compression_ratio'] = (
    train_df['summary_words_count'] / train_df['count_words']
)

In [ ]:
train_df['compression_ratio'].describe()

По данным видно что есть текст в котором сумаризация составляет 77% от длины исходного,посмотрим на этот текст

In [ ]:
train_df[train_df['compression_ratio'] == max(train_df['compression_ratio'])]

### **Вывод**: Медианный compression_ratio составляет около 0.18, то есть типичное summary содержит примерно 18% от количества слов исходного диалога. Для центральных 50% данных отношение находится примерно в диапазоне 0.156–0.206. Максимальное значение достигает 0.776, однако ручная проверка показала, что этот случай связан с коротким исходным диалогом (49 слов) и подробным, но корректным summary (38 слов), поэтому пример не удалялся.

In [ ]:
plt.hist(train_df['count_words'], bins=50)
plt.ylabel('Колличество примеров')
plt.xlabel('Длинна диалога в словах')
plt.title('Распределение длинны диалога в словах')
plt.show()

Основная масса диалогов относительно короткая, распределение скошено вправо, присутствует небольшой правый хвост длинных диалогов

## 04 — Non-ML baseline

Возьмем для базовой модели в качестве суммаризации первое и последнее предложение текста

In [ ]:
from nltk.tokenize import sent_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")

In [ ]:
line = test_df['dialogue']


Разобьем наш текст по предложениям и возьмем первое и последнее для каждого примера


In [ ]:
line_test = line.iloc[0]

In [ ]:
example = sent_tokenize(line_test)

In [ ]:
first = example[0] + example[-1]

In [ ]:
first

In [ ]:
first.replace('#Person1#:', '').replace('#Person2#:', '')

In [ ]:
example

In [ ]:
def create_baseline(dialogue):
  example = sent_tokenize(dialogue)
  first = example[0] + ' ' + example[-1]
  return first.replace('#Person1#:', '').replace('#Person2#:', '')



In [ ]:
test_df['baseline'] = test_df['dialogue'].apply(create_baseline)

In [ ]:
test_df['baseline']

## 05 — BLEU/ROUGE evaluation
Сохранён исходный расчёт **на train_df**. В исходном выводе выборка ошибочно называлась тестовой. BLEU здесь — `evaluate.load("bleu")`, не SacreBLEU; шкала 0–1. Outputs EDA/baseline воспроизведены выполнением в новой среде. ROUGE использует bootstrap, поэтому последние знаки могут отличаться от исходного вывода.

Так как наши тестовые данные имеют несколько вариантов summary, нам нужно сгруппировать примеры по столбцву 'dialgue' для корректной оценки качества по метрикам BLEU/ROUGE

In [ ]:
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')

In [ ]:
test_grouped = test_df.groupby('dialogue').agg(
    summary=('summary',list),
    baseline=('baseline','first')
    ).reset_index()

In [ ]:
test_grouped.shape

In [ ]:
test_grouped[['summary', 'baseline']].head()

In [ ]:
rouge_result = rouge.compute(
    predictions=test_grouped['baseline'].tolist(),
    references=test_grouped['summary'].tolist()
)

print(rouge_result)

In [ ]:
bleu_result = bleu.compute(
    predictions=test_grouped['baseline'].tolist(),
    references=test_grouped['summary'].tolist()
)

print(bleu_result)

### **Вывод**

В качестве простого non-ML baseline использовалось объединение первого и последнего предложения каждого диалога. Такой подход не использует машинное обучение и служит отправной точкой для дальнейшего сравнения с LLM.

Для корректной оценки использовалась тестовая выборка `test_df`. Она содержит 500 уникальных диалогов, для каждого из которых имеется по три эталонных summary. При расчёте метрик все три эталонных варианта учитывались.

На тестовой выборке были получены следующие результаты: **ROUGE-1 = 0.1978, ROUGE-2 = 0.0695, ROUGE-L = 0.1736 и BLEU = 0.0619**.

Низкий BLEU и снижение precision от unigram к 4-gram показывают, что baseline имеет некоторое совпадение с эталонными summary на уровне отдельных слов, но значительно хуже совпадает с ними по более длинным последовательностям слов.

**Brevity penalty = 0.7833** и **length ratio = 0.80** показывают, что predictions baseline в целом короче эталонных summary, что дополнительно снижает BLEU.

Полученные результаты будут использоваться как исходная точка для дальнейшего сравнения с **SmolLM2-360M-Instruct без дообучения** и этой же моделью после **LoRA fine-tuning**. Все варианты будут оцениваться на одной и той же тестовой выборке для корректного сравнения.

In [ ]:
results = pd.DataFrame({
    'model': ['Non-ML Baseline'],
    'split': ['test'],
    'BLEU': [bleu_result['bleu']],
    'ROUGE-1': [rouge_result['rouge1']],
    'ROUGE-2': [rouge_result['rouge2']],
    'ROUGE-L': [rouge_result['rougeL']],
})
results

## 06 — vLLM setup
Offline Python API. `dtype="float16"` выбран для T4 (без BF16). `max_model_len=1024`, `max_num_seqs=1` и `gpu_memory_utilization=0.5` ограничивают память для одного короткого теста. `enforce_eager=True` отключает компиляцию/CUDA graphs для простого диагностического запуска; это не настройка максимальной производительности. `spawn` задан до инициализации CUDA, чтобы дочерний процесс не наследовал CUDA через fork.

In [ ]:
llm = LLM(
    model='HuggingFaceTB/SmolLM2-360M-Instruct',
    dtype='float16',
    max_model_len=2048,
    gpu_memory_utilization=0.5,
    seed=42
)
print('MODEL_ENGINE_LOADED_OK')

## 07 — Base LLM inference
Ровно один короткий тест. Полная генерация по test dataset не запускается.

In [ ]:
messages = [{'role': 'user', 'content': 'Summarize this dialogue in one sentence:\nAlice: Can we meet at the library at three?\nBob: Yes, see you there at three.'}]
sampling_params = SamplingParams(temperature=0, max_tokens=64)
outputs = llm.chat(messages, sampling_params=sampling_params, use_tqdm=False)
summary = outputs[0].outputs[0].text.strip()
assert summary, 'Empty model response'
print(summary)
print('ONE_INFERENCE_OK')

Подготовим наши данные для модели

In [ ]:
messages = []

for dialogue in test_grouped["dialogue"]:
    messages.append([
        {
            "role": "user",
            "content": (
                "Write a short summary of the dialogue below. "
                "Do not repeat the dialogue. "
                "Describe the main events in 1-2 sentences. "
                "Return only the summary.\n\n"
                f"Dialogue:\n{dialogue}"
            )
        }
    ])

In [ ]:
messages[0]

In [ ]:
outputs = llm.chat(
    messages, 
    sampling_params=sampling_params, 
    use_tqdm=True
)

In [ ]:
print("Количество запросов:", len(outputs))

In [ ]:
for i in range(3):
    print(f"--- EXAMPLE {i+1} ---")
    print("DIALOGUE:")
    print(test_grouped["dialogue"].iloc[i])
    print("\nREFERENCE:")
    print(test_grouped["summary"].iloc[i])
    print("\nMODEL:")
    print(outputs[i].outputs[0].text.strip())
    print()

In [ ]:
test_grouped["base_llm"] = [
    output.outputs[0].text.strip()
    for output in outputs
]

In [ ]:
print("Generated:", len(outputs))
print("Empty:", (test_grouped["base_llm"].str.len() == 0).sum())

In [ ]:
bleu_result_0 = bleu.compute(
    predictions=test_grouped["base_llm"].tolist(),
    references=test_grouped["summary"].tolist()
)

bleu_result_0

In [ ]:
rouge_result_0 = rouge.compute(
    predictions=test_grouped["base_llm"].tolist(),
    references=test_grouped["summary"].tolist()
)

rouge_result_0

In [ ]:
new_result = pd.DataFrame({
    'model': ['SmolLM2-360M-Instruct zero-shot'],
    'split': ['test'],
    'BLEU': [bleu_result_0['bleu']],
    'ROUGE-1': [rouge_result_0['rouge1']],
    'ROUGE-2': [rouge_result_0['rouge2']],
    'ROUGE-L': [rouge_result_0['rougeL']],
})

results = pd.concat(
    [results, new_result],
    ignore_index=True
)

results

**Вывод:**
Модель SmolLM2-360M-Instruct без дообучения показала более высокие значения BLEU и ROUGE по сравнению с простым non-ML baseline, основанным на первой и последней реплике диалога. Это говорит о том, что даже без fine-tuning модель лучше улавливает основное содержание диалога.  
При этом ручная проверка показала существенные проблемы: модель часто генерирует слишком длинные ответы, частично копирует исходный диалог, иногда искажает факты и допускает галлюцинации. Это подтверждается значением length_ratio = 2.41, то есть сгенерированные тексты в среднем значительно длиннее эталонных summary.  
Таким образом, zero-shot baseline оказался лучше простого эвристического подхода, но качество всё ещё недостаточно для задачи суммаризации. Поэтому следующим этапом является дообучение модели с помощью LoRA на датасете DialogSum.

## 08 — LoRA (SFT)
Перед обучением модели , нужно подготовить данные в соответствующем виде.

In [ ]:
messages_sft = []

for index, row in train_df.iterrows():
    message = [
        {
            "role": "user",
            "content": (
                "Write a short summary of the dialogue below. "
                "Do not repeat the dialogue. "
                "Describe the main events in 1-2 sentences. "
                "Return only the summary.\n\n"
                f"Dialogue:\n{row['dialogue']}"
            )
        },
        {
            "role": "assistant",
            "content": row["summary"]
        }
    ]

    messages_sft.append(message)

In [ ]:
len(messages_sft)

In [ ]:
messages_sft[0]

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "messages": messages_sft
})

In [ ]:
train_dataset[0]

Так же подготовим валидационную выборку для проверки eval_loss во время обучения

In [ ]:
valid_messages = []

for index, row in valid_df.iterrows():
    message = [
        {
            "role": "user",
            "content": (
                "Write a short summary of the dialogue below. "
                "Do not repeat the dialogue. "
                "Describe the main events in 1-2 sentences. "
                "Return only the summary.\n\n"
                f"Dialogue:\n{row['dialogue']}"
            )
        },
        {
            "role": "assistant",
            "content": row["summary"]
        }
    ]

    valid_messages.append(message)

In [ ]:
valid_dataset = Dataset.from_dict({
    "messages": valid_messages
})

valid_dataset[0]

In [ ]:
print(len(train_dataset))
print(len(valid_dataset))
print(train_dataset.column_names)

In [ ]:
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/SmolLM2-360M-Instruct",
    max_seq_length = 2048,
    dtype = 'bfloat16',
    load_in_4bit = False
)

In [ ]:
for name, module in model.named_modules():
    if "proj" in name:
        print(name)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
model.print_trainable_parameters()

In [ ]:
def formatting_prompts_func(examples):
    texts = []

    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return {"text": texts}

In [ ]:
train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True
)

valid_dataset = valid_dataset.map(
    formatting_prompts_func,
    batched=True
)

In [ ]:
print(train_dataset[0]["text"])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

args = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_steps=50,
    weight_decay=0.01,
    lr_scheduler_type="linear",

    bf16=True,
    fp16=False,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,

    seed=42,
    output_dir="outputs",
    report_to="none",

    max_length=2048,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    args=args
)

In [ ]:
print(train_dataset.column_names)
print(valid_dataset.column_names)
print(train_dataset[0]["text"][:500])

In [ ]:
stainer_stats = trainer.train()

In [ ]:
train_steps = []
train_losses = []

eval_steps = []
eval_losses = []

for item in trainer.state.log_history:
    if "loss" in item:
        train_steps.append(item["step"])
        train_losses.append(item["loss"])

    if "eval_loss" in item:
        eval_steps.append(item["step"])
        eval_losses.append(item["eval_loss"])

In [ ]:
print(train_steps[:5])
print(train_losses[:5])

print(eval_steps[:5])
print(eval_losses[:5])


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(train_steps, train_losses)
plt.xlabel("Step")
plt.ylabel("Train Loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(eval_steps, eval_losses)
plt.xlabel("Step")
plt.ylabel("Validation Loss")
plt.title("Validation Loss")
plt.grid(True)
plt.show()

In [ ]:
model.save_pretrained("smollm2-dialogsum-lora")
tokenizer.save_pretrained("smollm2-dialogsum-lora")

In [ ]:
log_history = pd.DataFrame(trainer.state.log_history)
log_history.to_csv("training_log.csv", index=False)

## **Вывод по обучению LoRA:**
Во время fine-tuning модели SmolLM2-360M-Instruct с помощью LoRA training loss в начале быстро снизился примерно с 2.28 до диапазона 1.3–1.4, после чего вышел на плато и продолжил колебаться около этого уровня. Validation loss также постепенно снижался на протяжении обучения.
Рост validation loss не наблюдался, поэтому явных признаков переобучения нет. Это говорит о том, что выбранные параметры обучения и одна эпоха оказались достаточными для адаптации модели к задаче суммаризации диалогов.
Дальше качество модели необходимо проверить на тестовой выборке и сравнить с zero-shot baseline по метрикам BLEU/ROUGE и вручную по качеству сгенерированных summary.

# **Inference**

In [ ]:
messages_lora = []

for dialogue in test_grouped["dialogue"]:
    messages_lora.append([{
        "role": "user",
        "content": (
            "Write a short summary of the dialogue below. "
            "Do not repeat the dialogue. "
            "Describe the main events in 1-2 sentences. "
            "Return only the summary.\n\n"
            f"Dialogue:\n{dialogue}"
        )
    }])

In [ ]:
from vllm.lora.request import LoRARequest

llm_lora = LLM(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    dtype="float16",
    max_model_len=2048,
    gpu_memory_utilization=0.5,
    enable_lora=True,
    max_lora_rank=16,
    seed=42,
)

In [ ]:
lora_request = LoRARequest(
    "dialogsum_lora",
    1,
    "smollm2-dialogsum-lora"
)

sampling_params = SamplingParams(
    temperature=0,
    max_tokens=64,
)

In [ ]:
outputs_lora = llm_lora.chat(
    messages_lora,
    sampling_params=sampling_params,
    lora_request=lora_request,
    use_tqdm=True,
)

In [ ]:
test_grouped["lora_llm"] = [
    output.outputs[0].text.strip()
    for output in outputs_lora
]

In [ ]:
for i in range(3):
    print("=" * 80)
    print("REFERENCE:")
    print(test_grouped.iloc[i]["summary"])
    print()
    print("LORA:")
    print(test_grouped.iloc[i]["lora_llm"])

Считаем качество модели по метрикам BLEU/ROUGE

In [ ]:
predictions_lora = test_grouped["lora_llm"].tolist()
references_lora = test_grouped["summary"].tolist()

In [ ]:
bleu_result_lora = bleu.compute(
    predictions=predictions_lora,
    references=references_lora
)

bleu_result_lora

In [ ]:
rouge_result_lora = rouge.compute(
    predictions=predictions_lora,
    references=references_lora
)

rouge_result_lora

In [ ]:
print("BLEU:", bleu_result_lora["bleu"])

print("ROUGE-1:", rouge_result_lora["rouge1"])
print("ROUGE-2:", rouge_result_lora["rouge2"])
print("ROUGE-L:", rouge_result_lora["rougeL"])
print("ROUGE-Lsum:", rouge_result_lora["rougeLsum"])

После fine-tuning с LoRA качество суммаризации заметно улучшилось по всем автоматическим метрикам. BLEU вырос с 0.089 до 0.297, ROUGE-1 — с 0.313 до 0.489, ROUGE-2 — с 0.103 до 0.235, ROUGE-L — с 0.244 до 0.406. Это показывает, что модель стала существенно лучше воспроизводить структуру и содержание эталонных summary. При этом ручная проверка показывает, что отдельные фактические ошибки и галлюцинации всё ещё встречаются, поэтому автоматические метрики стоит дополнять ручной оценкой и LLM-as-a-judge.

## Inference

In [ ]:
!pip install openai -q

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

In [ ]:
response = client.chat.completions.create(
    model="Qwen/Qwen3-8B",
    messages=[
        {"role": "user", "content": "Reply with only: OK"}
    ],
    temperature=0.6,
    max_tokens=200,
)

msg = response.choices[0].message

print("content:", msg.content)
print("reasoning:", getattr(msg, "reasoning_content", None))

In [ ]:
def judge_pair(dialogue, references, zero_shot, lora):

    references_text = "\n".join(
        f"Reference {i+1}: {ref}"
        for i, ref in enumerate(references)
    )

    prompt = f"""
/no_think

You are an evaluator of dialogue summarization systems.

Evaluate the two summaries based primarily on the ORIGINAL DIALOGUE.

ORIGINAL DIALOGUE:
{dialogue}

REFERENCE SUMMARIES:
{references_text}

SUMMARY A — ZERO-SHOT:
{zero_shot}

SUMMARY B — LoRA:
{lora}

Evaluate both summaries from 1 to 5.

Criteria:

1. factuality
Are all claims supported by the original dialogue?
Penalize hallucinated or incorrect facts.

2. coverage
Does the summary capture the important events?

3. conciseness
Is the summary short and focused?

4. overall
Overall quality.

Return ONLY JSON.
DO NOT provide reasoning.
DO NOT provide explanations.
DO NOT use markdown.

Format:

{{
  "zero_shot": {{
    "factuality": 1,
    "coverage": 1,
    "conciseness": 1,
    "overall": 1
  }},
  "lora": {{
    "factuality": 1,
    "coverage": 1,
    "conciseness": 1,
    "overall": 1
  }}
}}
"""

    response = client.chat.completions.create(
        model="Qwen/Qwen3-8B",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7,
        max_tokens=1500,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    message = response.choices[0].message

    print("content:", message.content)
    print("reasoning:", getattr(message, "reasoning_content", None))

    if message.content is None:
        raise ValueError(
            "Qwen не вернул final content. "
            "Проверь reasoning_content выше."
        )

    return json.loads(message.content)

In [ ]:
i = 0

result = judge_pair(
    dialogue=test_grouped.iloc[i]["dialogue"],
    references=test_grouped.iloc[i]["summary"],
    zero_shot=test_grouped.iloc[i]["base_llm"],
    lora=test_grouped.iloc[i]["lora_llm"]
)

result

In [ ]:
import time
import json
import os

results = []

N = 10

for i in range(N):

    print(f"Evaluating {i+1}/{N}")

    try:
        result = judge_pair(
            dialogue=test_grouped.iloc[i]["dialogue"],
            references=test_grouped.iloc[i]["summary"],
            zero_shot=test_grouped.iloc[i]["base_llm"],
            lora=test_grouped.iloc[i]["lora_llm"]
        )

        results.append({
            "index": i,
            "zero_shot_factuality": result["zero_shot"]["factuality"],
            "zero_shot_coverage": result["zero_shot"]["coverage"],
            "zero_shot_conciseness": result["zero_shot"]["conciseness"],
            "zero_shot_overall": result["zero_shot"]["overall"],

            "lora_factuality": result["lora"]["factuality"],
            "lora_coverage": result["lora"]["coverage"],
            "lora_conciseness": result["lora"]["conciseness"],
            "lora_overall": result["lora"]["overall"],
        })

        
        pd.DataFrame(results).to_csv(
            "judge_results.csv",
            index=False
        )

    except Exception as e:
        print(f"ERROR on example {i}: {e}")

    time.sleep(1)

In [ ]:
judge_df = pd.read_csv("judge_results.csv")

print(f"Оценено примеров: {len(judge_df)}")
judge_df

In [ ]:
summary = pd.DataFrame({
    "Zero-shot": [
        judge_df["zero_shot_factuality"].mean(),
        judge_df["zero_shot_coverage"].mean(),
        judge_df["zero_shot_conciseness"].mean(),
        judge_df["zero_shot_overall"].mean(),
    ],
    "LoRA": [
        judge_df["lora_factuality"].mean(),
        judge_df["lora_coverage"].mean(),
        judge_df["lora_conciseness"].mean(),
        judge_df["lora_overall"].mean(),
    ]
}, index=[
    "Factuality",
    "Coverage",
    "Conciseness",
    "Overall"
])

summary

In [ ]:
lora_wins = (
    judge_df["lora_overall"] >
    judge_df["zero_shot_overall"]
).sum()

zero_shot_wins = (
    judge_df["zero_shot_overall"] >
    judge_df["lora_overall"]
).sum()

ties = (
    judge_df["lora_overall"] ==
    judge_df["zero_shot_overall"]
).sum()

print("LoRA wins:", lora_wins)
print("Zero-shot wins:", zero_shot_wins)
print("Ties:", ties)

LLM-as-a-judge на 10 случайных тестовых примерах показал преимущество LoRA: в 7 случаях из 10 LoRA получила более высокую overall-оценку, в одном случае уступила, а в двух случаях оценки совпали. Средние оценки LoRA были выше zero-shot по всем четырём критериям. Результат подтверждает тенденцию, наблюдаемую по BLEU и ROUGE, однако из-за небольшого размера выборки judge-оценку следует рассматривать как дополнительный качественный анализ, а не как статистически устойчивую оценку всей тестовой выборки.

## **Итог работы**
В рамках работы была построена система суммаризации диалогов на датасете DialogSum. Сначала был проверен простой baseline, затем исходная модель SmolLM2-360M-Instruct без дополнительного обучения и после этого та же модель была дообучена с помощью LoRA.
Само дообучение оказалось эффективным: при обучении изменялось только около 2,34% параметров модели, что позволило адаптировать модель под задачу суммаризации без полного fine-tuning всех весов.
После обучения качество заметно выросло. Например, BLEU увеличился с 0,089 у zero-shot модели до 0,297 после LoRA, а ROUGE-L — с 0,244 до 0,406. Рост наблюдается по всем использованным BLEU/ROUGE-метрикам.
При ручной проверке также было видно, что после LoRA модель стала лучше формировать короткие summaries и чаще сохранять основную информацию из диалога. При этом полностью избавиться от ошибок не удалось: в отдельных случаях модель всё ещё добавляет информацию, которой не было в исходном диалоге.
Для дополнительной проверки был использован LLM-as-a-judge на 10 тестовых примерах. LoRA получила более высокие средние оценки по всем четырём критериям:
- factuality: 1,2 → 2,6
- coverage: 1,2 → 2,7
- conciseness: 1,3 → 3,1
- overall: 1,2 → 2,6

В 7 из 10 рассмотренных примеров judge оценил LoRA выше zero-shot, в одном случае выше оказался zero-shot, ещё в двух случаях оценки совпали.
Таким образом, проведённый эксперимент показывает, что даже относительно небольшую модель SmolLM2-360M-Instruct можно заметно адаптировать под конкретную задачу с помощью LoRA. При этом BLEU/ROUGE и LLM-as-a-judge показывают одно направление улучшения. Однако judge был проведён только на 10 примерах, поэтому его результаты следует воспринимать как дополнительную качественную проверку, а не как статистически устойчивую оценку всей тестовой выборки.
Главный практический результат работы — не только увеличение метрик, но и получение полного воспроизводимого pipeline:
подготовка данных → baseline → zero-shot inference → оценка → LoRA fine-tuning → inference обученной модели → автоматические метрики → LLM-as-a-judge → анализ результатов.